## Handling Missing Values in Pandas
This notebook covers the basics of handling missing values in pandas DataFrames, including detection, dropping, filling, and interpolation. It also discusses practical machine learning scenarios where handling missing values is crucial.

### Importing Libraries and Creating a Sample Dataset
We start by importing the necessary libraries and creating a sample dataset with missing values.

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank'],
    'Age': [25, np.nan, 28, np.nan, 30, 26],
    'Salary': [50000, 60000, np.nan, 55000, 70000, np.nan],
    'Department': ['Sales', 'IT', 'HR', np.nan, 'Sales', 'IT'],
    'YearsExp': [2, 5, np.nan, 3, 6, 4]
})

print("Dataset with missing values:")
print(df)
print("\n" + "="*60)

### Detecting Missing Values
To handle missing values, we first need to detect them. Pandas provides several methods for this purpose.

In [ ]:
print("\nMissing values (True = missing):")
print(df.isnull())

# Count missing values per column
missing_counts = df.isnull().sum()
print("\nMissing value counts per column:")
print(missing_counts)

# Percentage of missing values
missing_percent = (df.isnull().sum() / len(df)) * 100
print("\nPercentage missing:")
print(missing_percent)

# Visualize missing data
print("\nWhich rows have any missing values:")
print(df[df.isnull().any(axis=1)])

The output of `df.isnull()` is a boolean mask where `True` indicates a missing value. The `sum()` method counts the number of `True` values in each column, giving us the total number of missing values per column. The percentage of missing values is calculated by dividing the count of missing values by the total number of rows and multiplying by 100.

### Dropping Missing Values
Dropping missing values is a simple approach to handle missing data. However, it can result in loss of information if there are many missing values.

In [ ]:
# Drop rows with any missing value
df_dropped_any = df.dropna()
print("\nAfter dropna() (remove rows with ANY missing):")
print(df_dropped_any)
print(f"Shape: {df_dropped_any.shape} (lost {len(df) - len(df_dropped_any)} rows)")

# Drop rows with all missing values
df_dropped_all = df.dropna(how='all')
print("\nAfter dropna(how='all') (remove rows with ALL missing):")
print(df_dropped_all)

# Drop specific columns with too many missing values
df_dropped_col = df.dropna(axis=1, thresh=4)  # Keep columns with >= 4 non-null values
print("\nAfter dropping columns with < 4 non-null values:")
print(df_dropped_col)

# Drop rows where specific column is missing
df_dropped_specific = df.dropna(subset=['Age'])
print("\nAfter dropping rows where Age is missing:")
print(df_dropped_specific)

Dropping missing values can be done using the `dropna()` method. By default, it drops rows with any missing values. The `how='all'` parameter can be used to drop rows with all missing values. The `axis=1` parameter can be used to drop columns with too many missing values. The `thresh` parameter can be used to specify the minimum number of non-null values required to keep a column. The `subset` parameter can be used to specify the columns to consider when dropping rows.

### Filling Missing Values
Filling missing values is another approach to handle missing data. It involves replacing missing values with imputed values.

In [ ]:
# Forward fill: propagate last known value forward
df_ffill = df.fillna(method='ffill')  # fillna works on copy
print("\nForward fill (ffill):")
print(df_ffill)

# Backward fill: propagate next known value backward
df_bfill = df.fillna(method='bfill')
print("\nBackward fill (bfill):")
print(df_bfill)

# Fill with constant value
df_constant = df.fillna({'Age': df['Age'].mean(), 'Department': 'Unknown', 'Salary': 0})
print("\nFill with constant values:")
print(df_constant)

# Fill with mean (numeric columns)
df_mean = df.copy()
df_mean['Age'] = df_mean['Age'].fillna(df_mean['Age'].mean())
df_mean['Salary'] = df_mean['Salary'].fillna(df_mean['Salary'].mean())
print("\nFill numeric columns with mean:")
print(df_mean)

# Fill with median
df_median = df.copy()
df_median['Age'] = df_median['Age'].fillna(df_median['Age'].median())
print("\nFill with median:")
print(df_median)

# Fill with mode (most common value)
df_mode = df.copy()
df_mode['Department'] = df_mode['Department'].fillna(df_mode['Department'].mode()[0])
print("\nFill categorical with mode:")
print(df_mode)

# Fill by group
df_group_fill = df.copy()
df_group_fill['Salary'] = df_group_fill.groupby('Department')['Salary'].transform(
    lambda x: x.fillna(x.mean())
)
print("\nFill salary with department-wise mean:")
print(df_group_fill)

Filling missing values can be done using the `fillna()` method. It can be used to fill missing values with a constant value, the mean, median, or mode of the column. The `method` parameter can be used to specify the filling method, such as 'ffill' for forward fill or 'bfill' for backward fill. The `groupby` method can be used to fill missing values by group.

### Interpolation
Interpolation is a method of filling missing values by estimating them based on the values of neighboring observations.

In [ ]:
# Linear interpolation
df_interp = df.copy()
df_interp['Age'] = df_interp['Age'].interpolate(method='linear')
print("\nLinear interpolation on Age:")
print(df_interp)

# Polynomial interpolation
df_poly = df.copy()
df_poly['Age'] = df_poly['Age'].interpolate(method='polynomial', order=2)
print("\nPolynomial interpolation (order=2):")
print(df_poly)

Interpolation can be done using the `interpolate()` method. The `method` parameter can be used to specify the interpolation method, such as 'linear' or 'polynomial'. The `order` parameter can be used to specify the order of the polynomial interpolation.

### Practical Machine Learning Scenarios
Handling missing values is an important step in machine learning pipelines. Here are some practical scenarios where handling missing values is crucial.

In [ ]:
# Scenario 1: Strategy depends on missingness type
df_research = df.copy()
for col in df_research.columns:
    missing_pct = (df_research[col].isnull().sum() / len(df_research)) * 100
    print(f"{col}: {missing_pct:.1f}% missing")
    
    if missing_pct > 30:
        print(f"  → Consider dropping {col}")
    elif df_research[col].dtype in ['float64', 'int64']:
        print(f"  → Fill with median")
    else:
        print(f"  → Fill with mode or custom value")

# Scenario 2: Prepare data for model training
df_clean = df.copy()
# Strategy: Drop columns with >30% missing, fill others with median/mode
df_clean = df_clean.drop(columns=['YearsExp'])  # 33% missing
df_clean['Age'].fillna(df_clean['Age'].median(), inplace=True)
df_clean['Salary'].fillna(df_clean['Salary'].median(), inplace=True)
df_clean['Department'].fillna('Unknown', inplace=True)

print("\nCleaned dataset ready for modeling:")
print(df_clean)
print("\nMissing values remain:", df_clean.isnull().sum().sum())

# Scenario 3: Add indicator for was-missing (informative feature)
df_with_indicator = df.copy()
df_with_indicator['Age_WasMissing'] = df_with_indicator['Age'].isnull().astype(int)
df_with_indicator['Age'].fillna(df_with_indicator['Age'].mean(), inplace=True)
print("\nWith 'was missing' indicator feature:")
print(df_with_indicator[['Age', 'Age_WasMissing']])

## Key Takeaways
* Always inspect: `isnull().sum()` and `isnull().sum()/len(df)` for %
* Drop if < 5% missing or column > 50% missing.
* Fill numeric: with mean/median (median more robust).
* Fill categorical: with mode or 'Unknown'.
* Interpolate: for time series with ordered missing patterns.
* Group-aware filling: use group statistics when appropriate.
* Consider adding 'was_missing' indicators as features for models.